# Анализ и предобработка датасета CoNLL-2003 для задачи NER

Этот ноутбук посвящен загрузке, первичному анализу и предобработке датасета CoNLL-2003, который используется для задачи распознавания именованных сущностей (NER). Цель - подготовить данные для обучения различных моделей машинного обучения.

## 1. Загрузка датасета CoNLL-2003 из локальных файлов

Датасет CoNLL-2003 представлен в виде текстовых файлов (`eng.train`, `eng.testa`, `eng.testb`) в формате CoNLL. В этом формате каждое слово и его метки (включая NER) находятся на отдельной строке, а предложения разделены пустыми строками.

Для парсинга этих файлов используется следующая функция:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def parse_conll_file(filepath):
    """
    Парсит файл в формате CoNLL и возвращает список предложений.
    Каждое предложение представлено словарем с ключами 'tokens' и 'ner_tags'.
    """
    sentences = []
    tokens = []
    ner_tags = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:  # Пустая строка - конец предложения
                if tokens:  # Добавляем предложение, если оно не пустое
                    sentences.append({'tokens': tokens, 'ner_tags': ner_tags})
                tokens = []
                ner_tags = []
            else:
                # Ожидаемый формат: word POS chunk NER
                parts = line.split()
                if len(parts) >= 4:  # Убедимся, что строка содержит как минимум 4 колонки
                    tokens.append(parts[0])
                    ner_tags.append(parts[3])
    # Добавляем последнее предложение, если файл не заканчивается пустой строкой
    if tokens:
        sentences.append({'tokens': tokens, 'ner_tags': ner_tags})
    return sentences


In [ ]:
# Загрузка данных из локальных файлов
train_data = parse_conll_file('data/raw/eng.train')
val_data = parse_conll_file('data/raw/eng.testa')  # 'testa' используется как валидационная выборка
test_data = parse_conll_file('data/raw/eng.testb')  # 'testb' используется как тестовая выборка

print(f"Загружено предложений для тренировки: {len(train_data)}")
print(f"Загружено предложений для валидации: {len(val_data)}")
print(f"Загружено предложений для теста: {len(test_data)}")


## 2. Преобразование тегов сущностей в ID

Для обучения моделей машинного обучения строковые метки сущностей (например, 'B-PER', 'O') необходимо преобразовать в числовые идентификаторы (ID). Определим список всех уникальных тегов сущностей в CoNLL-2003 и создадим словари для прямого и обратного преобразования.

In [ ]:
# Определим список всех уникальных тегов сущностей в CoNLL-2003
# Порядок важен, особенно для тега 'O' (обычно 0)
ner_tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
tag_to_id = {tag: i for i, tag in enumerate(ner_tag_names)}
id_to_tag = {i: tag for tag, i in tag_to_id.items()}

print(f"Список тегов и их ID: {tag_to_id}")

def map_tags_to_ids(sentences, tag_to_id_map):
    """
    Преобразует строковые теги NER в числовые ID для каждого предложения.
    """
    for sentence in sentences:
        sentence['ner_tags_ids'] = [tag_to_id_map[tag] for tag in sentence['ner_tags']]
    return sentences

# Применяем преобразование к загруженным данным
train_data = map_tags_to_ids(train_data, tag_to_id)
val_data = map_tags_to_ids(val_data, tag_to_id)
test_data = map_tags_to_ids(test_data, tag_to_id)

print("\nПример записи после преобразования тегов в ID:")
print(train_data[0])


## 3. Первичный анализ данных (EDA)

После загрузки и преобразования тегов проведем первичный разведочный анализ данных (EDA), чтобы понять их основные характеристики, такие как количество токенов и распределение типов сущностей.

### 3.1. Количество токенов в каждой выборке

In [ ]:
train_tokens = sum(len(s['tokens']) for s in train_data)
val_tokens = sum(len(s['tokens']) for s in val_data)
test_tokens = sum(len(s['tokens']) for s in test_data)

print(f"\nКоличество токенов в тренировочной выборке: {train_tokens}")
print(f"Количество токенов в валидационной выборке: {val_tokens}")
print(f"Количество токенов в тестовой выборке: {test_tokens}")


### 3.2. Распределение типов именованных сущностей

Визуализация распределения NER-тегов (без учета тега 'O', который обозначает отсутствие сущности) помогает понять баланс классов.

In [ ]:
all_ner_tags_ids = []
for dataset_split in [train_data, val_data, test_data]:
    for sentence in dataset_split:
        # Исключаем тег 'O' (ID 0) для анализа распределения только сущностей
        all_ner_tags_ids.extend([tag_id for tag_id in sentence['ner_tags_ids'] if tag_id != 0])

# Преобразуем ID обратно в строковые метки для подсчета и визуализации
all_ner_tags_names = [id_to_tag[tag_id] for tag_id in all_ner_tags_ids]
tag_counts = pd.Series(all_ner_tags_names).value_counts()

print("\nРаспределение типов именованных сущностей (без тега 'O'):")
print(tag_counts)

# Визуализация распределения
plt.figure(figsize=(10, 6))
sns.barplot(x=tag_counts.index, y=tag_counts.values, palette='viridis')
plt.title('Распределение типов именованных сущностей в CoNLL-2003 (без тега O)')
plt.xlabel('Тип сущности')
plt.ylabel('Количество')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 3.3. Анализ распределения длин предложений

Понимание распределения длин предложений важно для определения оптимальной максимальной длины последовательности для моделей.

In [ ]:
train_sentence_lengths = [len(s['tokens']) for s in train_data]
val_sentence_lengths = [len(s['tokens']) for s in val_data]
test_sentence_lengths = [len(s['tokens']) for s in test_data]

plt.figure(figsize=(12, 6))
sns.histplot(train_sentence_lengths, bins=50, kde=True, color='skyblue', label='Train')
sns.histplot(val_sentence_lengths, bins=50, kde=True, color='lightcoral', label='Validation')
sns.histplot(test_sentence_lengths, bins=50, kde=True, color='lightgreen', label='Test')
plt.title('Распределение длин предложений в CoNLL-2003')
plt.xlabel('Длина предложения (количество токенов)')
plt.ylabel('Частота')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nСредняя длина предложения (Train): {np.mean(train_sentence_lengths):.2f}")
print(f"Медианная длина предложения (Train): {np.median(train_sentence_lengths):.2f}")
print(f"Максимальная длина предложения (Train): {np.max(train_sentence_lengths)}")


### 3.4. Анализ частоты встречаемости токенов

Анализ наиболее частых токенов дает представление о лексическом составе датасета.

In [ ]:
from collections import Counter

all_tokens = []
for dataset_split in [train_data, val_data, test_data]:
    for sentence in dataset_split:
         all_tokens.extend(sentence['tokens'])

token_counts = Counter(all_tokens)
print("\nТоп-20 самых частых токенов:")
print(token_counts.most_common(20))


## 4. Паддинг и усечение последовательностей

Для подготовки данных к подаче в модели машинного обучения, особенно нейронные сети, необходимо привести все последовательности токенов и их соответствующих тегов к одинаковой длине. Мы будем использовать паддинг (дополнение) более коротких последовательностей специальным токеном/тегом и усечение более длинных последовательностей до максимальной длины.

Установим максимальную длину последовательности в 128 токенов, основываясь на анализе распределения длин предложений.

In [ ]:
MAX_SEQ_LENGTH = 128
PAD_TOKEN = "[PAD]"  # Специальный токен для паддинга
PAD_TAG_ID = tag_to_id['O']  # Используем ID тега 'O' для паддинга тегов

def pad_and_truncate_sequence(sequence, max_length, pad_value):
    """Паддинг или усечение последовательности до max_length."""
    if len(sequence) > max_length:
        return sequence[:max_length]
    elif len(sequence) < max_length:
        return sequence + [pad_value] * (max_length - len(sequence))
    return sequence

def preprocess_for_padding(sentences, max_length, pad_token, pad_tag_id):
    """Применяет паддинг и усечение к токенам и тегам."""
    processed_sentences = []
    for sentence in sentences:
        padded_tokens = pad_and_truncate_sequence(sentence['tokens'], max_length, pad_token)
        padded_tags_ids = pad_and_truncate_sequence(sentence['ner_tags_ids'], max_length, pad_tag_id)
        processed_sentences.append({
            'tokens': padded_tokens,
            'ner_tags_ids': padded_tags_ids,
            'original_length': len(sentence['tokens'])  # Сохраняем оригинальную длину
        })
    return processed_sentences

# Применяем паддинг и усечение к данным
train_data_padded = preprocess_for_padding(train_data, MAX_SEQ_LENGTH, PAD_TOKEN, PAD_TAG_ID)
val_data_padded = preprocess_for_padding(val_data, MAX_SEQ_LENGTH, PAD_TOKEN, PAD_TAG_ID)
test_data_padded = preprocess_for_padding(test_data, MAX_SEQ_LENGTH, PAD_TOKEN, PAD_TAG_ID)

print(f"\nПример обработанной записи после паддинга/усечения (Train):")
print(train_data_padded[0])
print(f"Длина токенов: {len(train_data_padded[0]['tokens'])}")
print(f"Длина ID тегов: {len(train_data_padded[0]['ner_tags_ids'])}")


## 5. Векторизация токенов с использованием BERT

Для использования моделей на основе трансформеров, таких как BERT, необходимо преобразовать токены в формат, понятный модели. Это включает токенизацию с использованием специального токенизатора модели и получение контекстуализированных эмбеддингов.

In [ ]:
from transformers import BertTokenizer, TFBertModel, BertTokenizerFast
import tensorflow as tf

# Загрузка токенизатора и модели BERT
# Используем 'bert-base-cased' - базовая модель BERT с учетом регистра
tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')
bert_model = TFBertModel.from_pretrained('bert-base-cased')

def tokenize_and_align_labels(sentences, tokenizer, max_length, tag_to_id):
    """
    Токенизация предложений с использованием токенизатора BERT
    и выравнивание меток NER с токенами BERT (с учетом суб-единиц).
    """
    tokenized_inputs = tokenizer(
        [s['tokens'] for s in sentences],
        is_split_into_words=True,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_overflowing_tokens=False,
        return_offsets_mapping=False,  # Не используем offset_mapping для простоты выравнивания меток
        return_token_type_ids=False,  # Не используем token_type_ids для этой задачи
        return_attention_mask=True,
        return_tensors="tf"  # Возвращаем TensorFlow тензоры
    )

    labels = []
    for i, sentence in enumerate(sentences):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Специальные токены (CLS, SEP, PAD) имеют word_idx None
            if word_idx is None:
                label_ids.append(PAD_TAG_ID)  # Присваиваем им ID паддинга (тег 'O')
            elif word_idx != previous_word_idx:
                # Начало нового слова
                label_ids.append(sentence['ner_tags_ids'][word_idx])
            else:
                # Суб-единица того же слова
                # Для суб-единиц внутри слова используем тот же тег, но меняем B- на I-
                tag_id = sentence['ner_tags_ids'][word_idx]
                if id_to_tag[tag_id].startswith('B-'):
                    i_tag = 'I-' + id_to_tag[tag_id][2:]
                    label_ids.append(tag_to_id[i_tag])
                else:
                    label_ids.append(tag_id)

            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = tf.constant(labels, dtype=tf.int64)
    return tokenized_inputs

# Применяем токенизацию и выравнивание меток к данным
# Используем оригинальные данные (не padded), так как токенизатор BERT сам выполнит паддинг/усечение
train_encoded = tokenize_and_align_labels(train_data, tokenizer, MAX_SEQ_LENGTH, tag_to_id)
val_encoded = tokenize_and_align_labels(val_data, tokenizer, MAX_SEQ_LENGTH, tag_to_id)
test_encoded = tokenize_and_align_labels(test_data, tokenizer, MAX_SEQ_LENGTH, tag_to_id)

print("\nПример токенизированных и закодированных данных (Train):")
print(train_encoded['input_ids'][0])
print(train_encoded['labels'][0])
print(f"Длина input_ids: {len(train_encoded['input_ids'][0])}")
print(f"Длина labels: {len(train_encoded['labels'][0])}")


## 6. Выводы по предобработке и гипотезы о методах NER

На основе проведенного анализа данных можно сделать выводы о необходимых шагах дальнейшей предобработки и сформулировать гипотезы о наиболее подходящих моделях для задачи NER на этом датасете.

### 6.1. Выполненные шаги предобработки

На текущий момент выполнены следующие ключевые шаги предобработки, которые подготовили данные для использования в моделях:

*   **Загрузка и парсинг данных:** Данные из файлов CoNLL-2003 успешно загружены и преобразованы в структурированный формат.
*   **Преобразование тегов сущностей в ID:** Строковые метки NER преобразованы в числовые идентификаторы (`tag_to_id`), что необходимо для большинства моделей машинного обучения.
*   **Паддинг и усечение последовательностей:** Последовательности токенов и тегов приведены к единой максимальной длине (`MAX_SEQ_LENGTH = 128`) с использованием паддинга, что важно для пакетной обработки, особенно в нейронных сетях.
*   **BERT-токенизация и выравнивание меток:** Данные токенизированы с использованием `BertTokenizerFast`, и метки NER выровнены с суб-единицами токенов BERT, что является обязательным шагом для моделей на основе трансформеров.


### 6.2. Статистика датасета CoNLL-2003

*   **Разделение данных:** Датасет CoNLL-2003 разделен на три части: обучающую (`eng.train`), валидационную (`eng.testa`) и тестовую (`eng.testb`).
    *   Количество предложений в обучающей выборке: 14987.
    *   Количество предложений в валидационной выборке: 3466.
    *   Количество предложений в тестовой выборке: 3684.
*   **Количество токенов:**
    *   Общее количество токенов в обучающей выборке: 204567.
    *   Общее количество токенов в валидационной выборке: 51578.
    *   Общее количество токенов в тестовой выборке: 46666.
*   **Статистика по NER-тегам:**
    *   Всего в датасете используется 9 типов NER-тегов (включая 'O'): `['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']`.
    *   Распределение именованных сущностей (без учета тега 'O') представлено на графике в разделе 3.2. Наиболее частыми типами сущностей являются `B-LOC`, `B-PER`, `B-ORG`, а наименее частыми – `I-MISC`, `I-LOC`.
    *   Анализ всех тегов показывает значительный дисбаланс классов: тег 'O' (не сущность) является преобладающим. Этот дисбаланс необходимо будет учитывать при обучении моделей, возможно, с помощью взвешенных функций потерь или специфических метрик.
*   **Характеристики предложений:**
    *   Анализ длин предложений (график в разделе 3.3) показывает, что большинство предложений имеют длину от 5 до 20 токенов.
    *   Для обучающей выборки: средняя длина предложения составляет 13.65 токенов, медианная – 9.00 токенов, а максимальная – 113 токенов.
    *   Выбранная максимальная длина последовательности `MAX_SEQ_LENGTH = 128` является разумным компромиссом между сохранением информации и вычислительной эффективностью.
*   **Анализ токенов:**
    *   Общее количество уникальных токенов во всем датасете составляет 23624.
    *   Топ-20 самых частых токенов включают: `[(',', 10876), ('.', 10874), ('the', 10672), ('of', 5426), ('in', 5073), ('to', 5067), ('a', 4414), ('(', 4226), (')', 4225), ('and', 4180), ('"', 3239), ('on', 3041), ('said', 2690), ("'s", 2283), ('for', 2048), ('-', 1866), ('1', 1845), ('The', 1611), ('was', 1593), ('-DOCSTART-', 1393)]`. Это в основном служебные слова, знаки препинания и наиболее общие слова английского языка.


### 6.3. Гипотезы о методах МО для NER и их обоснование

Учитывая последовательную природу задачи NER, характеристики датасета CoNLL-2003, выявленные в ходе EDA (распределение длин предложений, дисбаланс тегов, анализ токенов), а также успешный опыт применения различных подходов, перспективными методами для сравнения являются:

*   **Conditional Random Fields (CRF):** Классический статистический метод, хорошо подходящий для задач разметки последовательностей. CRF моделируют условную вероятность последовательности тегов при условии входной последовательности токенов, учитывая зависимости между соседними тегами. Это соответствует последовательной природе задачи NER. CRF могут служить хорошей базовой моделью для сравнения.

*   **Рекуррентные нейронные сети (RNN) с LSTM/GRU и CRF слоем:** Модели на основе RNN (особенно с LSTM или GRU ячейками) способны эффективно обрабатывать последовательности и улавливать контекст. Добавление CRF слоя на выходной слой такой сети (архитектура Bi-LSTM-CRF) позволяет учитывать зависимости между соседними тегами на этапе декодирования, что часто улучшает качество разметки. Последовательная природа задачи NER, где тег текущего слова сильно зависит от тегов предыдущих слов, а также необходимость учета локального контекста, что подтверждается анализом длин предложений (средняя длина 13.65 токенов), делают модели типа Bi-LSTM-CRF перспективными.

*   **Модели на основе трансформеров (BERT, RoBERta и др.):** Современные модели, такие как BERT, используют механизм внимания для улавливания как локальных, так и долгосрочных зависимостей в тексте, предоставляя мощные контекстуализированные эмбеддинги. Значительный размер словаря, наличие редких слов и потенциальная важность долгосрочных зависимостей в тексте для корректного распознавания сущностей, а также успешное применение предобученных трансформерных моделей для NER в современных исследованиях, указывают на целесообразность использования моделей типа BERT. Проведенная токенизация с помощью `BertTokenizerFast` и выравнивание меток уже подготовили данные для этого подхода.

В рамках Выпускной Квалификационной Работы планируется реализовать и сравнить несколько из этих подходов, чтобы оценить их эффективность на датасете CoNLL-2003, учитывая выявленные характеристики данных.
